# Transform Catalog References in Genie Spaces

This notebook applies catalog mapping to exported Genie Space configurations before deployment.

**What it does:**
1. Scans exported space JSON files for catalog references
2. Generates `catalog_mapping.csv` if not exists
3. Applies catalog replacements to serialized_space
4. Writes transformed configs for deployment

**Run this BEFORE Tgt_02_Deploy_Genie_Spaces if catalogs differ between source and target.**

## Configuration

In [ ]:
import os
import json
import csv
from typing import Dict, Set, List

dbutils.widgets.text("volume_path", "", "Import Volume Path")
dbutils.widgets.text("source_catalog", "", "Source Catalog")
dbutils.widgets.text("target_catalog", "", "Target Catalog")

volume_path = dbutils.widgets.get("volume_path")
source_catalog = dbutils.widgets.get("source_catalog")
target_catalog = dbutils.widgets.get("target_catalog")

if not volume_path:
    raise ValueError("volume_path parameter is required")

print(f"Volume path: {volume_path}")
print(f"Source catalog: {source_catalog}")
print(f"Target catalog: {target_catalog}")

## Scan exported spaces for catalog references

In [ ]:
def extract_catalogs_from_space(config: dict) -> Set[str]:
    """Extract all catalog names from a space config."""
    catalogs = set()
    
    serialized = config.get("serialized_space", "")
    if not serialized:
        return catalogs
    
    try:
        data = json.loads(serialized)
    except json.JSONDecodeError:
        return catalogs
    
    # Extract from data_sources.tables
    data_sources = data.get("data_sources", {})
    tables = data_sources.get("tables", [])
    
    for table in tables:
        identifier = table.get("identifier", "")
        if identifier:
            parts = identifier.split(".")
            if len(parts) >= 1:
                catalogs.add(parts[0])
    
    return catalogs


exported_dir = os.path.join(volume_path, "exported")
if not os.path.exists(exported_dir):
    raise FileNotFoundError(f"Exported directory not found: {exported_dir}")

all_catalogs = set()
space_catalogs = {}

json_files = [f for f in os.listdir(exported_dir) if f.endswith(".json")]
print(f"Scanning {len(json_files)} exported space(s)...")

for filename in json_files:
    filepath = os.path.join(exported_dir, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        config = json.load(f)
    
    catalogs = extract_catalogs_from_space(config)
    space_catalogs[filename] = catalogs
    all_catalogs.update(catalogs)
    
    print(f"  {filename}: {catalogs if catalogs else 'No catalogs found'}")

print(f"\nUnique catalogs found: {sorted(all_catalogs)}")

## Load or generate catalog mapping

In [ ]:
mapping_path = os.path.join(volume_path, "catalog_mapping.csv")
catalog_mapping = {}

if os.path.exists(mapping_path):
    print(f"Loading existing mapping from {mapping_path}")
    with open(mapping_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            src = row.get("source_catalog", "").strip()
            tgt = row.get("target_catalog", "").strip()
            if src and tgt:
                catalog_mapping[src] = tgt
    print(f"Loaded mappings: {catalog_mapping}")
else:
    print(f"No existing mapping found. Generating from parameters...")
    
    # Use parameters if provided
    if source_catalog and target_catalog:
        catalog_mapping[source_catalog] = target_catalog
        print(f"Using parameter mapping: {source_catalog} -> {target_catalog}")
    
    # Generate template for all discovered catalogs
    print(f"\nGenerating catalog_mapping.csv template...")
    with open(mapping_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(["source_catalog", "target_catalog", "notes"])
        for cat in sorted(all_catalogs):
            target = catalog_mapping.get(cat, "")
            spaces_using = [fname for fname, cats in space_catalogs.items() if cat in cats]
            writer.writerow([cat, target, f"Used by: {', '.join(spaces_using)}"])
    print(f"Written to: {mapping_path}")

if not catalog_mapping:
    print("\n⚠️  No catalog mappings defined. Edit catalog_mapping.csv and re-run, or provide source_catalog/target_catalog parameters.")

## Apply catalog transformations

In [ ]:
def transform_serialized_space(serialized_space: str, catalog_mapping: Dict[str, str]) -> str:
    """Transform catalog references in serialized_space JSON."""
    if not serialized_space or not catalog_mapping:
        return serialized_space
    
    try:
        data = json.loads(serialized_space)
    except json.JSONDecodeError:
        return serialized_space
    
    # Transform data_sources.tables
    data_sources = data.get("data_sources", {})
    tables = data_sources.get("tables", [])
    
    for table in tables:
        identifier = table.get("identifier", "")
        if identifier:
            parts = identifier.split(".")
            if len(parts) >= 3:
                old_catalog = parts[0]
                if old_catalog in catalog_mapping:
                    parts[0] = catalog_mapping[old_catalog]
                    table["identifier"] = ".".join(parts)
    
    # Transform instructions (only if it's a string)
    instructions = data.get("instructions")
    if instructions and isinstance(instructions, str):
        for old_cat, new_cat in catalog_mapping.items():
            instructions = instructions.replace(f"{old_cat}.", f"{new_cat}.")
        data["instructions"] = instructions
    
    return json.dumps(data)


if not catalog_mapping:
    print("No mappings to apply. Skipping transformation.")
else:
    transformed_dir = os.path.join(volume_path, "transformed")
    os.makedirs(transformed_dir, exist_ok=True)
    
    transform_results = []
    
    for filename in json_files:
        filepath = os.path.join(exported_dir, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            config = json.load(f)
        
        original_serialized = config.get("serialized_space", "")
        transformed_serialized = transform_serialized_space(original_serialized, catalog_mapping)
        
        # Count changes
        changes = 0
        for old_cat, new_cat in catalog_mapping.items():
            if old_cat in original_serialized:
                changes += original_serialized.count(f"{old_cat}.")
        
        config["serialized_space"] = transformed_serialized
        config["_metadata"]["catalog_mapping_applied"] = catalog_mapping
        
        # Write to transformed directory
        output_path = os.path.join(transformed_dir, filename)
        with open(output_path, "w", encoding="utf-8") as f:
            json.dump(config, f, indent=2, ensure_ascii=False)
        
        transform_results.append({"file": filename, "changes": changes})
        print(f"  {filename}: {changes} replacement(s)")
    
    print(f"\nTransformed files written to: {transformed_dir}")

## Update exported directory for deployment

In [ ]:
import shutil

if catalog_mapping:
    # Backup original
    backup_dir = os.path.join(volume_path, "exported_original")
    if not os.path.exists(backup_dir):
        shutil.copytree(exported_dir, backup_dir)
        print(f"Original files backed up to: {backup_dir}")
    
    # Copy transformed to exported (for deployment)
    for filename in json_files:
        src = os.path.join(transformed_dir, filename)
        dst = os.path.join(exported_dir, filename)
        shutil.copy2(src, dst)
    
    print(f"\nTransformed files copied to {exported_dir} for deployment.")

## Summary

In [ ]:
print("=" * 60)
print("CATALOG TRANSFORMATION COMPLETE")
print("=" * 60)
print(f"Spaces processed: {len(json_files)}")
print(f"Catalogs found: {sorted(all_catalogs)}")
print(f"Mappings applied: {catalog_mapping}")
print(f"\nFiles:")
print(f"  - catalog_mapping.csv: {mapping_path}")
print(f"  - Original backup: {volume_path}/exported_original/")
print(f"  - Transformed: {volume_path}/exported/")
print("\nNext step: Run Tgt_02_Deploy_Genie_Spaces to deploy transformed spaces.")
print("=" * 60)